In [1]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
data_dir = '/kaggle/input/datasets/vilquer/wikitext-2'
train_path = os.path.join(data_dir, 'wiki.train.tokens')
valid_path = os.path.join(data_dir, 'wiki.valid.tokens')
test_path = os.path.join(data_dir, 'wiki.test.tokens')

def load_tokens(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read().split()

print("Loading raw text...")
train_tokens = load_tokens(train_path)
valid_tokens = load_tokens(valid_path)
test_tokens = load_tokens(test_path)

# ---------------------------------------------------------
# BULLETPROOF VOCABULARY BUILDER
# ---------------------------------------------------------
max_vocab_size = 30000 
counter = Counter(train_tokens)

# 1. Get the most common words
most_common_words = [word for word, count in counter.most_common(max_vocab_size)]

# 2. Build dictionary safely: Force <unk> to 0, and add the rest sequentially
vocab = {'<unk>': 0}
for word in most_common_words:
    if word not in vocab: # Prevents adding <unk> twice if it was in the text
        vocab[word] = len(vocab)

# 3. Set global vocab size
vocab_size = len(vocab) 
print(f"Final Vocabulary Size: {vocab_size} tokens")
print(f"Highest Token ID: {max(vocab.values())}") # This will safely be vocab_size - 1

Loading raw text...
Final Vocabulary Size: 30000 tokens
Highest Token ID: 29999


In [3]:
def encode(tokens):
    return [vocab.get(w, 0) for w in tokens]

class LMDataset(Dataset):
    def __init__(self, data_ids, seq_len):
        self.seq_len = seq_len
        num_samples = (len(data_ids) - 1) // seq_len
        self.data = data_ids[:num_samples * seq_len + 1]

    def __len__(self):
        return (len(self.data) - 1) // self.seq_len

    def __getitem__(self, idx):
        i = idx * self.seq_len
        x = self.data[i : i + self.seq_len]
        y = self.data[i + 1 : i + self.seq_len + 1]
        return x, y

batch_size = 32
train_seq_len = 128
extrapolate_seq_len = 256

print("Encoding tokens to tensors...")
train_ids = torch.tensor(encode(train_tokens), dtype=torch.long)
valid_ids = torch.tensor(encode(valid_tokens), dtype=torch.long)
test_ids = torch.tensor(encode(test_tokens), dtype=torch.long)

# Initialize DataLoaders
train_loader = DataLoader(LMDataset(train_ids, train_seq_len), batch_size=batch_size, shuffle=True, drop_last=True)
valid_loader = DataLoader(LMDataset(valid_ids, train_seq_len), batch_size=batch_size, shuffle=False, drop_last=True)
test_loader = DataLoader(LMDataset(test_ids, train_seq_len), batch_size=batch_size, shuffle=False, drop_last=True)

# DataLoader for Length Generalization Testing (Double Sequence Length)
valid_ext_loader = DataLoader(LMDataset(valid_ids, extrapolate_seq_len), batch_size=batch_size, shuffle=False, drop_last=True)

print(f"DataLoaders ready. Train batches: {len(train_loader)}")

Encoding tokens to tensors...
DataLoaders ready. Train batches: 500


In [4]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) 

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :], None 

class LearnedPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0).expand(x.size(0), -1)
        return x + self.pe(positions), None

class RotaryPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.d_model = d_model
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, x):
        t = torch.arange(x.size(1), device=x.device).type_as(self.inv_freq)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        return x, torch.cat((freqs, freqs), dim=-1)

def apply_rotary_pos_emb(q, k, freqs):
    cos = freqs.cos().unsqueeze(0).unsqueeze(0)
    sin = freqs.sin().unsqueeze(0).unsqueeze(0)
    def rotate_half(x):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat((-x2, x1), dim=-1)
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None, rotary_freqs=None):
        batch_size = q.size(0)
        
        # Memory contiguity fixes are applied here
        q = self.w_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2).contiguous()
        k = self.w_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2).contiguous()
        v = self.w_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2).contiguous()

        if rotary_freqs is not None:
            q, k = apply_rotary_pos_emb(q, k, rotary_freqs[..., :self.d_k])
            q, k = q.contiguous(), k.contiguous()

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.w_o(out), attn

In [5]:
class SwappablePETransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, pe_module):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pe_module = pe_module
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'mha': MultiHeadAttention(d_model, num_heads),
                'norm1': nn.LayerNorm(d_model),
                'ffn': nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)),
                'norm2': nn.LayerNorm(d_model),
                'dropout': nn.Dropout(0.1)
            }) for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        # Causal mask for language modeling
        mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0).to(x.device)
        
        x = self.embedding(x)
        x, rotary_freqs = self.pe_module(x)
        
        for layer in self.layers:
            attn_out, _ = layer['mha'](x, x, x, mask, rotary_freqs)
            x = layer['norm1'](x + layer['dropout'](attn_out))
            ffn_out = layer['ffn'](x)
            x = layer['norm2'](x + layer['dropout'](ffn_out))
            
        return self.lm_head(x)

In [6]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(x).view(-1, vocab_size)
        loss = criterion(outputs, y.view(-1))
        loss.backward()
        
        # Gradient clipping prevents exploding gradients in early transformer training
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x).view(-1, vocab_size)
            loss = criterion(outputs, y.view(-1))
            total_loss += loss.item()
            
    loss = total_loss / len(loader)
    ppl = math.exp(loss) if loss < 20 else float('inf')
    return loss, ppl

In [7]:
# Model Hyperparameters
d_model, num_heads, num_layers, d_ff = 256, 8, 5, 1024
num_epochs = 8 # Set the number of epochs here

models = {
    #"Sinusoidal": SwappablePETransformer(vocab_size, d_model, num_heads, num_layers, d_ff, SinusoidalPE(d_model)).to(device),
    #"Learned": SwappablePETransformer(vocab_size, d_model, num_heads, num_layers, d_ff, LearnedPE(d_model)).to(device),
    "RoPE": SwappablePETransformer(vocab_size, d_model, num_heads, num_layers, d_ff, RotaryPE(d_model)).to(device)
}

criterion = nn.CrossEntropyLoss()

# Dictionary to store loss history for easy plotting later
history = {name: {'train': [], 'val': []} for name in models.keys()}

for name, model in models.items():
    print(f"\n======================================")
    print(f" TESTING: {name} Positional Encoding")
    print(f"======================================")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01) 
    
    # Pretraining phase over multiple epochs
    for epoch in range(num_epochs): 
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_ppl = evaluate(model, valid_loader, criterion)
        
        # Save to history tracking
        history[name]['train'].append(train_loss)
        history[name]['val'].append(val_loss)
        
        # Print epoch-wise metrics
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Perplexity: {val_ppl:.2f}")
    
    # Baseline Test 
    test_loss, test_ppl = evaluate(model, test_loader, criterion)
    print(f"--> Standard Test Set   | Loss: {test_loss:.4f} | Perplexity: {test_ppl:.2f}")
    
    # Length Generalization Test
    try:
        extrap_loss, extrap_ppl = evaluate(model, valid_ext_loader, criterion)
        print(f"--> Extrapolation (2x)  | Loss: {extrap_loss:.4f} | Perplexity: {extrap_ppl:.2f}")
    except Exception as e:
        print(f"--> Extrapolation (2x) FAILED: {str(e)}")


 TESTING: RoPE Positional Encoding
Epoch 1/8 | Train Loss: 6.4641 | Val Loss: 5.7447 | Val Perplexity: 312.53
Epoch 2/8 | Train Loss: 5.6562 | Val Loss: 5.4334 | Val Perplexity: 228.92
Epoch 3/8 | Train Loss: 5.2906 | Val Loss: 5.2543 | Val Perplexity: 191.40
Epoch 4/8 | Train Loss: 5.0181 | Val Loss: 5.1440 | Val Perplexity: 171.41
Epoch 5/8 | Train Loss: 4.7957 | Val Loss: 5.0830 | Val Perplexity: 161.26
Epoch 6/8 | Train Loss: 4.6028 | Val Loss: 5.0345 | Val Perplexity: 153.62
Epoch 7/8 | Train Loss: 4.4305 | Val Loss: 5.0184 | Val Perplexity: 151.17
Epoch 8/8 | Train Loss: 4.2735 | Val Loss: 5.0179 | Val Perplexity: 151.10
--> Standard Test Set   | Loss: 4.9464 | Perplexity: 140.66
--> Extrapolation (2x)  | Loss: 5.0187 | Perplexity: 151.21


In [8]:
import torch
# This saves only the weights
torch.save(model.state_dict(), "custom_model_weights.pth")
# OPTIONAL: Save the whole model (weights + architecture)
torch.save(model, "full_model.pth")